# LRao batch-size × cutoff ablation (IID multi)

Batch size is a hidden LRao-specific regularizer: the LFI loss inverts a
**per-batch** covariance of the score, so the batch sets the quality of the
Σ̂ the objective sees. This notebook sweeps it against the cutoff.

**Grid.** cutoff ∈ [1e-3, 1e-5, none] × batch ∈ [256, 512, 1024, 2048] ×
n ∈ [100, 200, 2000, 4000] × seeds (default 42–44; extend to 5 later —
resume-safe). 1000 epochs, no wd/clip, robust IQR front-end, [128] ReLU,
published pools/planting, Pd@Pfa=0.1.

**Per run we record three models/epochs:**
- **best-val** — global argmin of the validation LFI cost (10% split, checked
  every epoch); checkpoint saved. The deployable selection rule.
- **best-detection** — argmax test Pd over per-50-epoch snapshots (oracle
  diagnostic, metrics only). The question: does best-val predict it?
- **final** — epoch 1000; checkpoint saved.

**Budget:** 36 configs × seeds; epoch cost scales with n (not batch), n=4000
dominates. ≈ **4–5 h per seed on a T4** (~13 h for 3 seeds; ~4 h on an A100).
Resume-safe: interrupt and rerun the sweep cell; analysis works on partials.

In [ ]:
!git clone -b rebuttal --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import os, sys, json, time, copy, contextlib
import numpy as np
import torch
sys.path.insert(0, '.')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
assert os.path.exists('repro/data/pavia-u.mat')

In [ ]:
# ----------------- knobs -----------------
N_LIST   = [100, 200, 2000, 4000]
BATCHES  = [256, 512, 1024, 2048]
CUTOFFS  = [1e-3, 1e-5, 0.0]           # 0.0 = no regularization
SEEDS    = [42, 43, 44]                # extend to [42..46] later (resume-safe)
MAX_EPOCHS = 1000
SNAP = 50
VAL_FRACTION = 0.1
OUT = 'results/lrao_batch'

def clab(c):
    return 'none' if c == 0.0 else f'{c:.0e}'

# published multi reference (Pd@0.1) at the overlapping n
REF = {100: {'DART': 0.390, 'LRao': 0.501},
       200: {'DART': 0.402, 'LRao': 0.509},
       2000: {'DART': 0.595, 'LRao': 0.553}}

In [ ]:
# ----------------- protocol + self-contained LRao math -----------------
import yaml
from tqdm.auto import tqdm
from repro.protocols.iid import load_hsi, build_pools, _pd_at_fa, _auc
from repro.core.data import plant_targets
from repro.core.models import ScoreNet
from repro.core.normalization import robust_whitening_iqr

cfg = yaml.safe_load(open('repro/configs/iid_multi.yaml'))
cfg.update(dataset='repro/data/pavia-u.mat')


def build_data(seed):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    data, gt = load_hsi(cfg['dataset'])
    bkg, tgt = build_pools(data, gt.flatten(), cfg, 'multi')
    s = tgt.mean(axis=0).astype(np.float32)
    idx = np.arange(len(bkg)); rng.shuffle(idx)
    shuf = bkg[idx]
    assert len(shuf) >= max(N_LIST) + int(cfg['test_size'])
    pool = shuf[:max(N_LIST)].astype(np.float32)
    te = shuf[-int(cfg['test_size']):].astype(np.float32)
    planted, labels, _ = plant_targets(te, s, cfg['amplitude'],
                                       cfg['target_fraction'],
                                       model='additive', seed=seed)
    return pool, planted.astype(np.float32), labels, s


def lfi_loss(model, batch, cutoff, detach_sigma=True):
    n = len(batch)
    ctx = torch.no_grad() if detach_sigma else contextlib.nullcontext()
    with ctx:
        psi0 = model(batch)
        mu = psi0.mean(0)
        c = psi0 - mu
        Sigma = (c.T @ c) / max(n - 1, 1)
        U, S, Vh = torch.linalg.svd(Sigma)
        thr = float(cutoff) * S[0]
        S_inv = torch.where(S > thr, 1.0 / S, torch.zeros_like(S))
        Sigma_inv = Vh.T @ torch.diag(S_inv) @ U.T
    from torch.func import jacrev, vmap
    J = vmap(jacrev(lambda x: model(x.unsqueeze(0)).squeeze(0)))(batch)
    G = J.mean(0)
    return -(G.T @ Sigma_inv @ G).trace()


@torch.no_grad()
def lrao_score(model, train_data, test_data, s, cutoff, delta=0.01):
    model.eval()
    d = train_data.shape[1]
    X_tr = torch.tensor(train_data, dtype=torch.float32, device=DEVICE)
    X_te = torch.tensor(test_data, dtype=torch.float32, device=DEVICE)
    I_d = torch.eye(d, device=DEVICE)
    psi_tr = model(X_tr).cpu().numpy().astype(np.float64)
    if not np.all(np.isfinite(psi_tr)):
        return np.zeros(len(test_data))
    mu = psi_tr.mean(0)
    Sigma = (psi_tr - mu).T @ (psi_tr - mu) / max(len(train_data) - 1, 1)
    U, S, Vh = np.linalg.svd((Sigma + Sigma.T) / 2)
    thr = float(cutoff) * S[0]
    S_inv = np.where(S > thr, 1.0 / S, 0.0)
    Sigma_inv = Vh.T @ np.diag(S_inv) @ U.T
    G = np.zeros((psi_tr.shape[1], d))
    for j in range(d):
        plus = model(X_tr + delta * I_d[j]).cpu().numpy()
        minus = model(X_tr - delta * I_d[j]).cpu().numpy()
        G[:, j] = ((plus - minus) / (2.0 * delta)).mean(0)
    g_s = G @ np.asarray(s, np.float64)
    J_s = float(g_s @ Sigma_inv @ g_s)
    psi_te = model(X_te).cpu().numpy()
    return (psi_te - mu) @ (Sigma_inv @ g_s) / np.sqrt(max(J_s, 1e-12))


def metrics(labels, sc):
    return dict(pd=_pd_at_fa(labels, sc, cfg['pfa']), auc=_auc(labels, sc))

In [ ]:
# ----------------- trainer: best-val + best-detection + final -----------------
def train_one(tr, cutoff, bs_req, seed, label, planted, labels, s, ckpt_dir):
    torch.manual_seed(seed)
    W = robust_whitening_iqr(tr)
    net = ScoreNet(tr.shape[1], [128], 'relu', whitening=W).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=cfg['lr'], weight_decay=0.0)
    X = torch.tensor(tr).to(DEVICE)
    n_val = max(1, int(len(X) * VAL_FRACTION))
    perm0 = torch.randperm(len(X))
    Xf, Xv = X[perm0[:-n_val]], X[perm0[-n_val:]]
    N = len(Xf); bs = min(int(bs_req), N)
    best_val, bv_state, bv_epoch = float('inf'), None, 0
    val_hist, snaps = [], []
    pbar = tqdm(range(1, MAX_EPOCHS + 1), desc=label, leave=False)
    for ep in pbar:
        net.train()
        perm = torch.randperm(N)
        for i in range(0, N, bs):
            try:
                loss = lfi_loss(net, Xf[perm[i:i + bs]], cutoff)
            except Exception:
                continue
            if not torch.isfinite(loss):
                continue
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        try:
            vl = float(lfi_loss(net, Xv, cutoff).detach())
        except Exception:
            vl = float('nan')
        val_hist.append(vl)
        if np.isfinite(vl) and vl < best_val:
            best_val, bv_epoch = vl, ep
            bv_state = copy.deepcopy(net.state_dict())
        if ep % SNAP == 0:
            sc = lrao_score(net, tr, planted, s, cutoff)
            snaps.append({'epoch': ep, **metrics(labels, sc)})
            pbar.set_postfix(pd=f"{snaps[-1]['pd']:.3f}", bv=bv_epoch)
    final_state = copy.deepcopy(net.state_dict())
    os.makedirs(ckpt_dir, exist_ok=True)
    torch.save({'state_dict': {k: v.cpu() for k, v in bv_state.items()},
                'epoch': bv_epoch},
               os.path.join(ckpt_dir, label + '_bestval.pt'))
    torch.save({'state_dict': {k: v.cpu() for k, v in final_state.items()},
                'epoch': MAX_EPOCHS},
               os.path.join(ckpt_dir, label + '_final.pt'))
    best_snap = max(snaps, key=lambda t: t['pd'])
    out = {'bv_epoch': bv_epoch, 'bd_epoch': best_snap['epoch'],
           'bd': {'pd': best_snap['pd'], 'auc': best_snap['auc']},
           'val_hist': [round(v, 6) for v in val_hist], 'snaps': snaps}
    for kind, state in (('bv', bv_state), ('final', final_state)):
        net.load_state_dict(state); net.eval()
        sc = lrao_score(net, tr, planted, s, cutoff)
        out[kind if kind == 'final' else 'bv_model'] = metrics(labels, sc)
    return out

In [ ]:
# ----------------- the sweep (resume-safe) -----------------
os.makedirs(OUT, exist_ok=True)
met_path = os.path.join(OUT, 'metrics.json')
rec = json.load(open(met_path)) if os.path.exists(met_path) else {}
rec['_meta'] = dict(n_list=N_LIST, batches=BATCHES, seeds=SEEDS,
                    cutoffs=[clab(c) for c in CUTOFFS],
                    max_epochs=MAX_EPOCHS, val_fraction=VAL_FRACTION)
t0 = time.time()
for seed in SEEDS:
    pool, planted, labels, s = build_data(seed)
    for n in N_LIST:
        tr = pool[:n]
        for bs in BATCHES:
            for co in CUTOFFS:
                key = f'c{clab(co)}_b{bs}_n{n}_s{seed}'
                if rec.get(key, {}).get('final'):
                    continue
                t1 = time.time()
                r = train_one(tr, co, bs, seed, key, planted, labels, s,
                              os.path.join(OUT, 'ckpt'))
                r['sec'] = round(time.time() - t1)
                rec[key] = r
                json.dump(rec, open(met_path, 'w'))
                print(f"{key}: bv ep{r['bv_epoch']} Pd={r['bv_model']['pd']:.3f}"
                      f" | bd ep{r['bd_epoch']} Pd={r['bd']['pd']:.3f}"
                      f" | final Pd={r['final']['pd']:.3f} ({r['sec']}s)",
                      flush=True)
print(f'TOTAL {(time.time() - t0) / 3600:.2f} h')

In [ ]:
# ----------------- analysis + figures (partial-safe) -----------------
import matplotlib.pyplot as plt
from IPython.display import Image, display

FIG = os.path.join(OUT, 'figures'); os.makedirs(FIG, exist_ok=True)
rec = json.load(open(met_path))
CL = [clab(c) for c in CUTOFFS]

def get(cl, bs, n, sd):
    return rec.get(f'c{cl}_b{bs}_n{n}_s{sd}')

def show(fig, name):
    fig.tight_layout()
    p = os.path.join(FIG, name + '.png')
    fig.savefig(p, dpi=200); fig.savefig(p.replace('.png', '.pdf'))
    plt.close(fig); display(Image(p, width=980))

# --- fig 1: Pd (best-val model) vs batch, panel per n, lines per cutoff ---
fig, axes = plt.subplots(1, len(N_LIST), figsize=(4.0 * len(N_LIST), 4.0),
                         sharey=True, squeeze=False)
cols = plt.cm.viridis(np.linspace(0, 0.8, len(CL)))
for a, n in zip(axes[0], N_LIST):
    for i, cl in enumerate(CL):
        m = [np.nanmean([r['bv_model']['pd'] for sd in SEEDS
                         if (r := get(cl, bs, n, sd))] or [np.nan])
             for bs in BATCHES]
        a.plot(BATCHES, m, 'o-', color=cols[i], lw=1.6, label=f'cutoff {cl}')
        mf = [np.nanmean([r['final']['pd'] for sd in SEEDS
                          if (r := get(cl, bs, n, sd))] or [np.nan])
              for bs in BATCHES]
        a.plot(BATCHES, mf, 'o:', color=cols[i], lw=1, alpha=0.6)
    if n in REF:
        a.axhline(REF[n]['LRao'], color='k', ls='--', lw=1, label='LRao (pub)')
        a.axhline(REF[n]['DART'], color='tab:red', ls=':', lw=1,
                  label='DART (pub)')
    a.set_xscale('log', base=2); a.set_xticks(BATCHES)
    a.set_xticklabels(BATCHES); a.minorticks_off()
    a.set_xlabel('batch size'); a.set_title(f'n={n}'); a.grid(alpha=0.3)
axes[0][0].set_ylabel('Pd@0.1 (solid=best-val, dotted=final)')
axes[0][-1].legend(fontsize=7)
fig.suptitle('LRao: batch size x cutoff (mean over seeds)')
show(fig, 'batch_cutoff_pd')

# --- fig 2: best-val epoch vs best-detection epoch (correlation) ---
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
mark = {256: 'o', 512: 's', 1024: '^', 2048: 'D'}
xs, ys = [], []
for i, cl in enumerate(CL):
    for bs in BATCHES:
        for n in N_LIST:
            for sd in SEEDS:
                r = get(cl, bs, n, sd)
                if not r:
                    continue
                axes[0].scatter(r['bv_epoch'], r['bd_epoch'], s=22,
                                color=cols[i], marker=mark[bs], alpha=0.6)
                xs.append(r['bv_epoch']); ys.append(r['bd_epoch'])
                axes[1].scatter(r['bv_model']['pd'], r['bd']['pd'], s=22,
                                color=cols[i], marker=mark[bs], alpha=0.6)
lim = [0, MAX_EPOCHS]
axes[0].plot(lim, lim, 'k--', lw=1)
if len(xs) > 2:
    axes[0].set_title('epochs: best-val vs best-detection '
                      f'(r={np.corrcoef(xs, ys)[0, 1]:.2f})')
axes[0].set_xlabel('best-VAL epoch'); axes[0].set_ylabel('best-DETECTION epoch')
axes[0].grid(alpha=0.3)
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlabel('Pd of best-val model')
axes[1].set_ylabel('Pd at best-detection snapshot (oracle)')
axes[1].set_title('what val selection costs vs the oracle')
axes[1].grid(alpha=0.3)
for i, cl in enumerate(CL):
    axes[0].scatter([], [], color=cols[i], label=f'cutoff {cl}')
for bs in BATCHES:
    axes[0].scatter([], [], color='gray', marker=mark[bs], label=f'b={bs}')
axes[0].legend(fontsize=6.5, ncol=2)
show(fig, 'val_vs_detection_correlation')

# --- summary ---
lines = ['# LRao batch x cutoff — Pd@0.1 (best-val model, mean over seeds)', '']
for n in N_LIST:
    lines += [f'## n={n}',
              '| cutoff | ' + ' | '.join(f'b={b}' for b in BATCHES) + ' |',
              '|' + '---|' * (len(BATCHES) + 1)]
    for cl in CL:
        vals = [np.nanmean([r['bv_model']['pd'] for sd in SEEDS
                            if (r := get(cl, bs, n, sd))] or [np.nan])
                for bs in BATCHES]
        lines.append(f'| {cl} | ' + ' | '.join(
            f'{v:.3f}' if np.isfinite(v) else '—' for v in vals) + ' |')
    lines.append('')
open(os.path.join(OUT, 'summary.md'), 'w').write('\n'.join(lines))
print('\n'.join(lines))

In [ ]:
def lfi_loss_fd(model, batch, cutoff, delta, detach_sigma=True):
    """Original formulation: G by central differences with step delta."""
    n, d = batch.shape
    with (torch.no_grad() if detach_sigma else contextlib.nullcontext()):
        psi0 = model(batch); mu = psi0.mean(0); c = psi0 - mu
        Sigma = (c.T @ c) / max(n - 1, 1)
        U, S, Vh = torch.linalg.svd(Sigma)
        S_inv = torch.where(S > float(cutoff) * S[0], 1.0 / S,
                            torch.zeros_like(S))
        Sigma_inv = Vh.T @ torch.diag(S_inv) @ U.T
    I_d = torch.eye(d, device=batch.device); dt = float(delta)
    Xp = (batch.unsqueeze(0) + dt * I_d.unsqueeze(1)).reshape(-1, d)
    Xm = (batch.unsqueeze(0) - dt * I_d.unsqueeze(1)).reshape(-1, d)
    G = ((model(Xp).reshape(d, n, -1).mean(1)
          - model(Xm).reshape(d, n, -1).mean(1)) / (2 * dt)).T
    return -(G.T @ Sigma_inv @ G).trace()


# ==== SCENARIO VERIFICATION: LRao new recipe vs DART (embedded refs) ====
# LRao: no cutoff, FULL batch, no ES, fd delta=0.01. 2 seeds, ~30-50 min T4.
SCEN = [('single_asphalt',   [1],    6, 2000, 0.972),
        ('single_bricks',    [8],    6, 1000, 0.940),
        ('mix2_asph_bricks', [1, 8], 6, 2000, 0.823),
        ('mix2_asph_trees',  [1, 4], 6, 2000, 0.590),   # control: LRao should win
        ('single_meadows_t4', [2],   4, 2000, 0.865)]
data_, gt_ = load_hsi(cfg['dataset'])
flat_ = data_.reshape(-1, data_.shape[-1]); g_ = gt_.flatten()
CLS = {c: flat_[g_ == c] for c in range(10)}
sc_path = os.path.join(OUT, 'scenario_metrics.json')
os.makedirs(OUT, exist_ok=True)
srec = json.load(open(sc_path)) if os.path.exists(sc_path) else {}
for name, classes, tcls, n, dart_ref in SCEN:
    sig = CLS[tcls].mean(axis=0).astype(np.float32)
    for seed in [42, 43]:
        key = f'{name}_s{seed}'
        if key in srec:
            continue
        t0 = time.time()
        rng = np.random.default_rng(seed); torch.manual_seed(seed)
        bkg = np.vstack([CLS[c] for c in sorted(classes)])
        idx = np.arange(len(bkg)); rng.shuffle(idx); shuf = bkg[idx]
        tr = shuf[:n].astype(np.float32)
        te = shuf[-2000:].astype(np.float32)
        planted, labels, _ = plant_targets(te, sig, cfg['amplitude'],
                                           cfg['target_fraction'],
                                           model='additive', seed=seed)
        planted = planted.astype(np.float32)
        torch.manual_seed(seed)
        net = ScoreNet(103, [128], 'relu',
                       whitening=robust_whitening_iqr(tr)).to(DEVICE)
        opt = torch.optim.Adam(net.parameters(), lr=cfg['lr'],
                               weight_decay=0.0)
        X = torch.tensor(tr).to(DEVICE)
        for ep in tqdm(range(1000), desc=key, leave=False):
            net.train()
            try:
                loss = lfi_loss_fd(net, X, 0.0, 0.01)
            except Exception:
                continue
            if torch.isfinite(loss):
                opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        sc = lrao_score(net, tr, planted, sig, cutoff=0.0, delta=0.01)
        srec[key] = {'pd': _pd_at_fa(labels, sc, cfg['pfa']),
                     'auc': _auc(labels, sc), 'sec': round(time.time() - t0)}
        json.dump(srec, open(sc_path, 'w'))
        print(f'{key}: LRao Pd={srec[key]["pd"]:.3f}  (DART: {dart_ref:.3f})',
              flush=True)
print(f"\n{'scenario':20s} {'LRao':>7} {'DART':>7}  verdict")
for name, classes, tcls, n, dart_ref in SCEN:
    lr_pd = np.mean([srec[f'{name}_s{s}']['pd'] for s in [42, 43]
                     if f'{name}_s{s}' in srec] or [np.nan])
    v = ('DART WINS' if dart_ref > lr_pd + 0.02 else
         ('LRao wins' if lr_pd > dart_ref + 0.02 else 'tie'))
    print(f'{name:20s} {lr_pd:7.3f} {dart_ref:7.3f}  {v}')

In [ ]:
def lfi_loss_fd(model, batch, cutoff, delta, detach_sigma=True):
    """Original formulation: G by central differences with step delta."""
    n, d = batch.shape
    with (torch.no_grad() if detach_sigma else contextlib.nullcontext()):
        psi0 = model(batch); mu = psi0.mean(0); c = psi0 - mu
        Sigma = (c.T @ c) / max(n - 1, 1)
        U, S, Vh = torch.linalg.svd(Sigma)
        S_inv = torch.where(S > float(cutoff) * S[0], 1.0 / S,
                            torch.zeros_like(S))
        Sigma_inv = Vh.T @ torch.diag(S_inv) @ U.T
    I_d = torch.eye(d, device=batch.device); dt = float(delta)
    Xp = (batch.unsqueeze(0) + dt * I_d.unsqueeze(1)).reshape(-1, d)
    Xm = (batch.unsqueeze(0) - dt * I_d.unsqueeze(1)).reshape(-1, d)
    G = ((model(Xp).reshape(d, n, -1).mean(1)
          - model(Xm).reshape(d, n, -1).mean(1)) / (2 * dt)).T
    return -(G.T @ Sigma_inv @ G).trace()


# ==== HEAD-TO-HEAD at large n (DART minibatch vs LRao full-batch) ====
from repro.core.models import dsm_loss
from repro.core.detectors import dsm_additive
from repro.core.data import Whitening
N_BIG = [1000, 4000, 8000]
SEEDS_BIG = [42, 43]
DART_CONFIGS = [('none', 0.01), ('none', 0.005), ('published', 0.05),
                ('none', 0.001), ('none', 0.0005)]
OUT_HH = 'results/headtohead'
os.makedirs(os.path.join(OUT_HH, 'ckpt'), exist_ok=True)

def zca(tr, floor):
    X = np.asarray(tr, np.float64); mu = X.mean(0); Xc = X - mu
    Sig = (Xc.T @ Xc) / max(len(X) - 1, 1); Sig = (Sig + Sig.T) / 2
    ev, V = np.linalg.eigh(Sig)
    if floor == 'none':
        inv = np.where(ev > 0, 1.0 / np.sqrt(np.abs(ev)), 0.0)
    else:
        inv = 1.0 / np.sqrt(np.clip(ev, max(float(ev[-1]) * 1e-5, 1e2), None))
    return Whitening(mu.astype(np.float32),
                     (V @ np.diag(inv) @ V.T).astype(np.float32))

def build_big(seed, max_n):
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    data, gt = load_hsi(cfg['dataset'])
    bkg, tgt = build_pools(data, gt.flatten(), cfg, 'multi')
    s = tgt.mean(axis=0).astype(np.float32)
    idx = np.arange(len(bkg)); rng.shuffle(idx); shuf = bkg[idx]
    need = max_n + max(64, int(max_n * 0.1))
    assert len(shuf) >= need + int(cfg['test_size'])
    pool = shuf[:need].astype(np.float32)
    te = shuf[-int(cfg['test_size']):].astype(np.float32)
    planted, labels, _ = plant_targets(te, s, cfg['amplitude'],
                                       cfg['target_fraction'],
                                       model='additive', seed=seed)
    return pool, planted.astype(np.float32), labels, s

hh_path = os.path.join(OUT_HH, 'metrics.json')
hh = json.load(open(hh_path)) if os.path.exists(hh_path) else {}
for seed in SEEDS_BIG:
    pool, planted, labels, s = build_big(seed, max(N_BIG))
    for n in N_BIG:
        tr = pool[:n]; val = pool[n:n + max(64, int(n * 0.1))]
        for fl, rho in DART_CONFIGS:
            key = f'DART_f{fl}_r{rho}_n{n}_s{seed}'
            if key in hh:
                continue
            t0 = time.time(); torch.manual_seed(seed)
            net = ScoreNet(103, [128], cfg['activation'],
                           whitening=zca(tr, fl)).to(DEVICE)
            opt = torch.optim.Adam(net.parameters(), lr=cfg['lr'],
                                   weight_decay=0.0)
            Xf = torch.tensor(tr).to(DEVICE)
            Xv = torch.tensor(val).to(DEVICE)
            sigma = float(np.sqrt(rho)); N = len(Xf)
            bv, bv_state = float('inf'), None
            for ep in tqdm(range(3000), desc=key, leave=False):
                net.train(); perm = torch.randperm(N)
                for i in range(0, N, 512):
                    loss = dsm_loss(net, Xf[perm[i:i + 512]], sigma)
                    opt.zero_grad(); loss.backward(); opt.step()
                net.eval()
                with torch.random.fork_rng(
                        devices=[DEVICE] if DEVICE == 'cuda' else []):
                    torch.manual_seed(12345)
                    with torch.no_grad():
                        vl = float(dsm_loss(net, Xv, sigma))
                if np.isfinite(vl) and vl < bv:
                    bv = vl; bv_state = copy.deepcopy(net.state_dict())
            net.load_state_dict(bv_state); net.eval()
            sc = dsm_additive(planted, tr, net, s)
            hh[key] = {'pd': _pd_at_fa(labels, sc, cfg['pfa']),
                       'auc': _auc(labels, sc), 'sec': round(time.time() - t0)}
            json.dump(hh, open(hh_path, 'w')); print(key, hh[key], flush=True)
        key = f'LRao_n{n}_s{seed}'
        if key not in hh:
            t0 = time.time(); torch.manual_seed(seed)
            net = ScoreNet(103, [128], 'relu',
                           whitening=robust_whitening_iqr(tr)).to(DEVICE)
            opt = torch.optim.Adam(net.parameters(), lr=cfg['lr'],
                                   weight_decay=0.0)
            Xf = torch.tensor(tr).to(DEVICE)
            Xv = torch.tensor(val).to(DEVICE)
            bv, bv_state = float('inf'), None
            for ep in tqdm(range(1000), desc=key, leave=False):
                net.train()
                try:
                    loss = lfi_loss_fd(net, Xf, 0.0, 0.01)
                except Exception:
                    continue
                if torch.isfinite(loss):
                    opt.zero_grad(); loss.backward(); opt.step()
                net.eval()
                try:
                    vl = float(lfi_loss_fd(net, Xv, 0.0, 0.01).detach())
                except Exception:
                    vl = float('nan')
                if np.isfinite(vl) and vl < bv:
                    bv = vl; bv_state = copy.deepcopy(net.state_dict())
            fin_state = copy.deepcopy(net.state_dict())
            out = {}
            for kind, st in (('final', fin_state), ('bv', bv_state)):
                net.load_state_dict(st); net.eval()
                sc = lrao_score(net, tr, planted, s, cutoff=0.0, delta=0.01)
                out[kind] = {'pd': _pd_at_fa(labels, sc, cfg['pfa']),
                             'auc': _auc(labels, sc)}
            hh[key] = {**out, 'sec': round(time.time() - t0)}
            json.dump(hh, open(hh_path, 'w')); print(key, hh[key], flush=True)

print(f"\n{'model':30s}" + ''.join(f'{n:>8}' for n in N_BIG))
for fl, rho in DART_CONFIGS:
    row = [np.nanmean([hh[f'DART_f{fl}_r{rho}_n{n}_s{sd}']['pd']
                       for sd in SEEDS_BIG
                       if f'DART_f{fl}_r{rho}_n{n}_s{sd}' in hh] or [np.nan])
           for n in N_BIG]
    print(f'DART {fl} rho={rho:<8}' + ''.join(f'{v:8.3f}' for v in row))
for kind in ('bv', 'final'):
    row = [np.nanmean([hh[f'LRao_n{n}_s{sd}'][kind]['pd']
                       for sd in SEEDS_BIG
                       if f'LRao_n{n}_s{sd}' in hh] or [np.nan])
           for n in N_BIG]
    print(f'LRao new ({kind}):{"":9s}' + ''.join(f'{v:8.3f}' for v in row))

In [ ]:
# ----------------- display all saved figures -----------------
import glob
from IPython.display import Image, display
for p in sorted(glob.glob('results/lrao_batch/figures/*.png')):
    print(p)
    display(Image(filename=p, width=980))

In [ ]:
# ----------------- zip -----------------
!zip -qr lrao_batch_light.zip results/lrao_batch results/headtohead -x "*/ckpt/*"
!zip -qr lrao_batch_full.zip results/lrao_batch results/headtohead
!ls -lh lrao_batch_*.zip
try:
    from google.colab import files
    files.download('lrao_batch_light.zip')
except Exception as e:
    print('manual download:', e)